# Zeek Multi-Log C2 Detection and Attribution

This notebook implements the selected Zeek feature set for three experiments:

1. Binary detection: benign vs C2.
2. Multiclass attribution: benign, Cobalt Strike, Havoc, Mythic, Sliver.
3. Unseen C2 generalisation: train without one C2 family, then test on it.

The modelling unit is a communication channel:

```text
dataset + src_ip + dst_ip + dst_port + proto
```

Raw infrastructure identifiers such as IP addresses, domain names, SNI values,
certificate subjects, and user-agent strings are used only for grouping or for
derived behavioural features. They are not used directly as model predictors.



## Optional package install



In [24]:
# %pip install pandas numpy scikit-learn matplotlib seaborn scipy



In [25]:
from __future__ import annotations

from collections import Counter
from functools import lru_cache
from itertools import combinations
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 120)

## Configuration



In [ ]:
BASE_DIR = Path(r"D:\ResearchPaper")
OUTPUT_DIR = BASE_DIR / "outputs" / "zeek_multilog_ml"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [
    BASE_DIR / "Beacons",
    BASE_DIR / "BenignTraffic",
]

LOG_TYPES = ["conn", "dns", "http", "ssl", "tls", "x509"]
C2_FAMILIES = ["cobalt_strike", "havoc", "mythic", "sliver"]
ALL_FAMILIES = ["benign", *C2_FAMILIES]

RANDOM_STATE = 42
TOP_K_FEATURES = 50

# Benign captures are highly imbalanced, so the split is fixed to keep enough
# benign volume and capture diversity in both training and testing.
FORCE_TRAIN_DATASETS = {
    "benign_Monday-WorkingHours",
    "benign_2013-12-17_capture1",
    "benign_CTU-Normal-20-2017-04-30_win-normal",
    "benign_amazon_https",
    "benign1_smashburger",
    "benign2_smashburger",
    "benign3_smashburger",
    "benign4_smashburger",
}

FORCE_TEST_DATASETS = {
    "benign_2017-05-01_normal",
    "benign_2015-03-24_capture1-only-dns",
    "benign_jquery_http",
    "benign5_smashburger",
    "benign6_smashburger",
    "benign7_smashburger",
    "benign8_smashburger",
}

# Use None for final runs. Set an integer for quick debugging.
MAX_ROWS_PER_LOG: int | None = None

# Optional per-log caps. Keep DNS uncapped for final reported results.
MAX_ROWS_PER_LOG_TYPE: dict[str, int | None] = {
    "dns": None,
}

## Shared helpers



In [27]:
def infer_family(path: Path) -> str:
    text = str(path).lower()
    if "benigntraffic" in text or "benign" in text or "normal" in text:
        return "benign"
    if "cobaltstrike" in text or "cobalt-strike" in text or "cobalt_strike" in text:
        return "cobalt_strike"
    if "havoc" in text:
        return "havoc"
    if "mythic" in text:
        return "mythic"
    if "sliver" in text:
        return "sliver"
    return "unknown"


def infer_dataset(path: Path) -> str:
    return path.parent.name


def infer_log_type(path: Path) -> str:
    return path.name.removesuffix(".log")


def discover_logs() -> pd.DataFrame:
    rows = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for log_type in LOG_TYPES:
            for path in root.rglob(f"{log_type}.log"):
                rows.append(
                    {
                        "family": infer_family(path),
                        "dataset": infer_dataset(path),
                        "log_type": infer_log_type(path),
                        "path": path,
                    }
                )
    return pd.DataFrame(rows).sort_values(["family", "dataset", "log_type"]).reset_index(drop=True)


def read_zeek_log(path: Path, max_rows: int | None = MAX_ROWS_PER_LOG) -> pd.DataFrame:
    fields = None
    rows = []
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            line = line.rstrip("\n")
            if line.startswith("#fields"):
                fields = line.split("\t")[1:]
                continue
            if line.startswith("#") or not line:
                continue
            if fields is None:
                continue
            rows.append(line.split("\t"))
            if max_rows is not None and len(rows) >= max_rows:
                break
    if fields is None:
        raise ValueError(f"No #fields header found in {path}")

    df = pd.DataFrame(rows, columns=fields)
    df = df.replace({"-": np.nan, "(empty)": np.nan})
    df["family"] = infer_family(path)
    df["dataset"] = infer_dataset(path)
    df["source_file"] = str(path)
    return df


def load_log_type(log_type: str, inventory: pd.DataFrame) -> pd.DataFrame:
    paths = list(inventory.loc[inventory["log_type"].eq(log_type), "path"])
    print(f"Loading {len(paths)} {log_type}.log files", flush=True)
    frames = []
    max_rows = MAX_ROWS_PER_LOG_TYPE.get(log_type, MAX_ROWS_PER_LOG)
    for i, path in enumerate(paths, start=1):
        frames.append(read_zeek_log(path, max_rows=max_rows))
        if i == 1 or i % 10 == 0 or i == len(paths):
            print(f"  loaded {i}/{len(paths)} {log_type}.log files", flush=True)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def to_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def safe_div(num: pd.Series, den: pd.Series) -> pd.Series:
    return (num / den.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)


@lru_cache(maxsize=250_000)
def _entropy_text_cached(text: str) -> float:
    if not text:
        return np.nan
    counts = Counter(text)
    total = len(text)
    return -sum((count / total) * math.log2(count / total) for count in counts.values())


@lru_cache(maxsize=250_000)
def _digit_ratio_cached(text: str) -> float:
    if not text:
        return np.nan
    return sum(ch.isdigit() for ch in text) / len(text)


@lru_cache(maxsize=250_000)
def _label_count_cached(text: str) -> float:
    text = text.strip(".")
    if not text:
        return np.nan
    return float(len(text.split(".")))


@lru_cache(maxsize=250_000)
def _max_label_length_cached(text: str) -> float:
    labels = [part for part in text.strip(".").split(".") if part]
    if not labels:
        return np.nan
    return float(max(len(part) for part in labels))


def entropy_text(value: object) -> float:
    if pd.isna(value):
        return np.nan
    return _entropy_text_cached(str(value))


def digit_ratio(value: object) -> float:
    if pd.isna(value):
        return np.nan
    return _digit_ratio_cached(str(value))


def label_count(value: object) -> float:
    if pd.isna(value):
        return np.nan
    return _label_count_cached(str(value))


def max_label_length(value: object) -> float:
    if pd.isna(value):
        return np.nan
    return _max_label_length_cached(str(value))


def list_count(value: object) -> float:
    if pd.isna(value):
        return 0.0
    text = str(value)
    if not text:
        return 0.0
    return float(len([part for part in text.split(",") if part]))


def list_mean(value: object) -> float:
    if pd.isna(value):
        return np.nan
    vals = pd.to_numeric(pd.Series(str(value).split(",")), errors="coerce").dropna()
    return float(vals.mean()) if len(vals) else np.nan


def list_std(value: object) -> float:
    if pd.isna(value):
        return np.nan
    vals = pd.to_numeric(pd.Series(str(value).split(",")), errors="coerce").dropna()
    return float(vals.std(ddof=1)) if len(vals) > 1 else 0.0


def mode_or_unknown(series: pd.Series) -> str:
    values = series.dropna().astype(str)
    if values.empty:
        return "unknown"
    return values.mode().iloc[0]


def coefficient_of_variation(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna().to_numpy()
    if len(values) < 2:
        return np.nan
    mean = values.mean()
    if mean == 0:
        return np.nan
    return values.std(ddof=1) / mean


def median_absolute_deviation(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna().to_numpy()
    if len(values) == 0:
        return np.nan
    median = np.median(values)
    return float(np.median(np.abs(values - median)))


inventory = discover_logs()
print(f"Discovered {len(inventory)} Zeek log files", flush=True)
inventory["rows"] = inventory["path"].map(lambda path: sum(1 for line in path.open("r", encoding="utf-8", errors="replace") if not line.startswith("#")))
inventory.to_csv(OUTPUT_DIR / "zeek_log_inventory.csv", index=False)
inventory.pivot_table(index="family", columns="log_type", values="path", aggfunc="count", fill_value=0)



Discovered 239 Zeek log files


log_type,conn,dns,http,ssl,x509
family,,,,,
benign,15,5,5,13,5
cobalt_strike,31,18,21,22,14
havoc,2,2,2,2,2
mythic,13,11,13,0,0
sliver,14,13,2,14,0


## Build `conn.log` channel features



In [28]:
conn = load_log_type("conn", inventory)
print(f"Loaded conn rows: {len(conn):,}", flush=True)


def build_conn_channel_features(conn_df: pd.DataFrame) -> pd.DataFrame:
    df = conn_df.rename(
        columns={
            "ts": "ts",
            "id.orig_h": "src_ip",
            "id.orig_p": "src_port",
            "id.resp_h": "dst_ip",
            "id.resp_p": "dst_port",
        }
    ).copy()

    numeric = [
        "ts",
        "src_port",
        "dst_port",
        "duration",
        "orig_bytes",
        "resp_bytes",
        "orig_pkts",
        "resp_pkts",
        "orig_ip_bytes",
        "resp_ip_bytes",
        "missed_bytes",
    ]
    df = to_numeric(df, numeric)
    for col in numeric:
        if col not in df.columns:
            df[col] = np.nan
    for col in ["proto", "service", "conn_state"]:
        if col not in df.columns:
            df[col] = "unknown"

    df["duration"] = df["duration"].fillna(0.0)
    for col in ["orig_bytes", "resp_bytes", "orig_pkts", "resp_pkts", "orig_ip_bytes", "resp_ip_bytes", "missed_bytes"]:
        df[col] = df[col].fillna(0.0)

    df["total_bytes_row"] = df["orig_bytes"] + df["resp_bytes"]
    df["total_pkts_row"] = df["orig_pkts"] + df["resp_pkts"]
    df["total_ip_bytes_row"] = df["orig_ip_bytes"] + df["resp_ip_bytes"]

    keys = ["family", "dataset", "src_ip", "dst_ip", "dst_port", "proto"]
    df = df.sort_values(keys + ["ts"])
    df["iat"] = df.groupby(keys, dropna=False)["ts"].diff()
    df["iat_group_median"] = df.groupby(keys, dropna=False)["iat"].transform("median")
    df["iat_abs_deviation"] = (df["iat"] - df["iat_group_median"]).abs()
    df["minute_bin"] = np.floor(df["ts"] / 60.0)

    channel = (
        df.groupby(keys, dropna=False)
        .agg(
            connections_window=("ts", "count"),
            start_ts=("ts", "min"),
            end_ts=("ts", "max"),
            service=("service", mode_or_unknown),
            conn_state=("conn_state", mode_or_unknown),
            duration_mean=("duration", "mean"),
            duration_median=("duration", "median"),
            duration_std=("duration", "std"),
            orig_bytes_sum=("orig_bytes", "sum"),
            resp_bytes_sum=("resp_bytes", "sum"),
            orig_bytes_mean=("orig_bytes", "mean"),
            resp_bytes_mean=("resp_bytes", "mean"),
            orig_pkts_sum=("orig_pkts", "sum"),
            resp_pkts_sum=("resp_pkts", "sum"),
            orig_ip_bytes_sum=("orig_ip_bytes", "sum"),
            resp_ip_bytes_sum=("resp_ip_bytes", "sum"),
            missed_bytes_sum=("missed_bytes", "sum"),
            mean_iat=("iat", "mean"),
            median_iat=("iat", "median"),
            std_iat=("iat", "std"),
            mad_iat=("iat_abs_deviation", "median"),
        )
        .reset_index()
    )

    channel["active_span_seconds"] = (channel["end_ts"] - channel["start_ts"]).clip(lower=0)
    channel["cv_iat"] = safe_div(channel["std_iat"], channel["mean_iat"])
    channel["total_bytes"] = channel["orig_bytes_sum"] + channel["resp_bytes_sum"]
    channel["total_packets"] = channel["orig_pkts_sum"] + channel["resp_pkts_sum"]
    channel["total_ip_bytes"] = channel["orig_ip_bytes_sum"] + channel["resp_ip_bytes_sum"]
    channel["orig_resp_byte_ratio"] = safe_div(channel["orig_bytes_sum"] + 1, channel["resp_bytes_sum"] + 1)
    channel["orig_resp_packet_ratio"] = safe_div(channel["orig_pkts_sum"] + 1, channel["resp_pkts_sum"] + 1)
    channel["bytes_per_packet"] = safe_div(channel["total_bytes"], channel["total_packets"])
    channel["bytes_per_second"] = safe_div(channel["total_bytes"], channel["active_span_seconds"])
    channel["packets_per_second"] = safe_div(channel["total_packets"], channel["active_span_seconds"])

    minute_counts = (
        df.groupby(keys + ["minute_bin"], dropna=False)
        .size()
        .rename("minute_connections")
        .reset_index()
    )
    burst = (
        minute_counts.groupby(keys, dropna=False)["minute_connections"]
        .agg(max_minute_connections="max", mean_minute_connections="mean")
        .reset_index()
    )
    burst["burstiness"] = safe_div(burst["max_minute_connections"], burst["mean_minute_connections"])
    channel = channel.merge(burst[keys + ["burstiness"]], on=keys, how="left")

    src_window = (
        df.groupby(["family", "dataset", "src_ip"], dropna=False)
        .agg(
            unique_destinations_window=("dst_ip", "nunique"),
            source_connections_window=("ts", "count"),
        )
        .reset_index()
    )
    channel = channel.merge(src_window, on=["family", "dataset", "src_ip"], how="left")
    channel["destination_reuse"] = safe_div(channel["connections_window"], channel["unique_destinations_window"])

    channel["is_c2"] = channel["family"].ne("benign").astype(int)
    return channel


channel_features = build_conn_channel_features(conn)
print(f"Built channel features: {channel_features.shape}", flush=True)
channel_features.shape



Loading 75 conn.log files
  loaded 1/75 conn.log files
  loaded 10/75 conn.log files
  loaded 20/75 conn.log files
  loaded 30/75 conn.log files
  loaded 40/75 conn.log files
  loaded 50/75 conn.log files
  loaded 60/75 conn.log files
  loaded 70/75 conn.log files
  loaded 75/75 conn.log files
Loaded conn rows: 842,353
Built channel features: (54200, 42)


(54200, 42)

## DNS features



In [29]:
dns = load_log_type("dns", inventory)
print(f"Loaded dns rows: {len(dns):,}", flush=True)


def build_dns_source_features(dns_df: pd.DataFrame) -> pd.DataFrame:
    if dns_df.empty:
        return pd.DataFrame(columns=["family", "dataset", "src_ip"])
    df = dns_df.rename(columns={"ts": "ts", "id.orig_h": "src_ip"}).copy()
    df = to_numeric(df, ["ts"])
    if "query" not in df.columns:
        df["query"] = np.nan
    df["query_length"] = df["query"].astype("string").str.len()
    df["query_labels"] = df["query"].map(label_count)
    df["query_max_label_length"] = df["query"].map(max_label_length)
    df["query_entropy"] = df["query"].map(entropy_text)
    df["query_digit_ratio"] = df["query"].map(digit_ratio)
    df["answer_count"] = df["answers"].map(list_count) if "answers" in df.columns else np.nan
    df["ttl_mean"] = df["TTLs"].map(list_mean) if "TTLs" in df.columns else np.nan
    df["ttl_std"] = df["TTLs"].map(list_std) if "TTLs" in df.columns else np.nan
    df["is_nxdomain"] = df["rcode_name"].eq("NXDOMAIN").astype(float) if "rcode_name" in df.columns else np.nan

    keys = ["family", "dataset", "src_ip"]
    df = df.sort_values(keys + ["ts"])
    df["dns_iat"] = df.groupby(keys, dropna=False)["ts"].diff()

    profile = (
        df.groupby(keys, dropna=False)
        .agg(
            dns_queries_window=("query", "count"),
            dns_unique_queries_window=("query", "nunique"),
            dns_query_length_mean=("query_length", "mean"),
            dns_query_length_max=("query_length", "max"),
            dns_label_count_mean=("query_labels", "mean"),
            dns_max_label_length_mean=("query_max_label_length", "mean"),
            dns_query_entropy_mean=("query_entropy", "mean"),
            dns_query_digit_ratio_mean=("query_digit_ratio", "mean"),
            dns_repeated_query_ratio=("query", lambda s: 1.0 - (s.nunique(dropna=True) / max(len(s.dropna()), 1))),
            dns_nxdomain_ratio=("is_nxdomain", "mean"),
            dns_answer_count_mean=("answer_count", "mean"),
            dns_ttl_mean=("ttl_mean", "mean"),
            dns_ttl_std=("ttl_std", "mean"),
            dns_mean_iat=("dns_iat", "mean"),
            dns_std_iat=("dns_iat", "std"),
            dns_qtype=("qtype_name", mode_or_unknown) if "qtype_name" in df.columns else ("query", lambda _s: "unknown"),
            dns_rcode=("rcode_name", mode_or_unknown) if "rcode_name" in df.columns else ("query", lambda _s: "unknown"),
        )
        .reset_index()
    )
    profile["dns_cv_iat"] = safe_div(profile["dns_std_iat"], profile["dns_mean_iat"])
    span = df.groupby(keys, dropna=False)["ts"].agg(["min", "max"]).reset_index()
    span["dns_span_minutes"] = ((span["max"] - span["min"]) / 60).clip(lower=1)
    profile = profile.merge(span[keys + ["dns_span_minutes"]], on=keys, how="left")
    profile["dns_queries_per_minute"] = safe_div(profile["dns_queries_window"], profile["dns_span_minutes"])
    return profile


dns_features = build_dns_source_features(dns)
print(f"Built DNS source features: {dns_features.shape}", flush=True)
dns_features.head()



Loading 49 dns.log files


  loaded 1/49 dns.log files
  loaded 10/49 dns.log files
  loaded 20/49 dns.log files
  loaded 30/49 dns.log files
  loaded 40/49 dns.log files
  loaded 49/49 dns.log files
Loaded dns rows: 529,366
Built DNS source features: (94, 23)


,family,dataset,src_ip,dns_queries_window,dns_unique_queries_window,dns_query_length_mean,dns_query_length_max,dns_label_count_mean,dns_max_label_length_mean,dns_query_entropy_mean,dns_query_digit_ratio_mean,dns_repeated_query_ratio,dns_nxdomain_ratio,dns_answer_count_mean,dns_ttl_mean,dns_ttl_std,dns_mean_iat,dns_std_iat,dns_qtype,dns_rcode,dns_cv_iat,dns_span_minutes,dns_queries_per_minute
0,benign,benign_2013-12-17_capture1,10.0.0.46,1075,721,25.48093,34,5.789767,7.168372,3.557547,0.356888,0.329302,0.251163,0.851163,15301.044870,353.922054,1.146876,3.964047,PTR,NOERROR,3.456388,20.529078,52.364749
1,benign,benign_2015-03-24_capture1-only-dns,10.0.0.34,3029,212,17.944866,42,3.016507,8.353582,3.315456,0.023403,0.930010,0.001321,3.031033,2541.518978,3457.861753,2.272384,8.713528,A,NOERROR,3.834532,114.679636,26.412710
2,benign,benign_2017-05-01_normal,10.0.2.15,21126,2685,21.727634,79,3.450488,9.869355,3.514791,0.051959,0.872905,0.001183,2.595948,3212.834778,3642.019221,0.330758,1.623177,A,NOERROR,4.907443,116.454472,181.409950
3,benign,benign_2017-05-01_normal,fe80::69dd:e614:2b2:dfd0,2,1,4.0,4,1.000000,4.000000,2.000000,0.000000,0.500000,0.000000,0.000000,NaN,NaN,0.106421,NaN,A,unknown,NaN,1.000000,2.000000
4,benign,benign_CTU-Normal-20-2017-04-30_win-normal,10.0.2.15,15085,1143,21.689095,71,3.422340,9.824395,3.507152,0.042880,0.924229,0.000133,2.310706,3405.502782,5081.016338,0.749178,3.292683,A,NOERROR,4.395059,188.343393,80.093067


## HTTP features



In [30]:
http = load_log_type("http", inventory)
print(f"Loaded http rows: {len(http):,}", flush=True)


def build_http_channel_features(http_df: pd.DataFrame) -> pd.DataFrame:
    if http_df.empty:
        return pd.DataFrame(columns=["family", "dataset", "src_ip", "dst_ip", "dst_port", "proto"])
    df = http_df.rename(
        columns={
            "ts": "ts",
            "id.orig_h": "src_ip",
            "id.resp_h": "dst_ip",
            "id.resp_p": "dst_port",
        }
    ).copy()
    df["proto"] = "tcp"
    df = to_numeric(df, ["ts", "dst_port", "status_code", "request_body_len", "response_body_len"])
    df["uri_length"] = df["uri"].astype("string").str.len() if "uri" in df.columns else np.nan
    df["host_length"] = df["host"].astype("string").str.len() if "host" in df.columns else np.nan
    df["user_agent_length"] = df["user_agent"].astype("string").str.len() if "user_agent" in df.columns else np.nan
    df["uri_entropy"] = df["uri"].map(entropy_text) if "uri" in df.columns else np.nan
    df["is_get"] = df["method"].eq("GET").astype(float) if "method" in df.columns else np.nan
    df["is_post"] = df["method"].eq("POST").astype(float) if "method" in df.columns else np.nan
    if "user_agent" in df.columns:
        ua_reuse = df.groupby(["dataset", "user_agent"], dropna=False)["user_agent"].transform("count")
        df["user_agent_reuse_count"] = ua_reuse
    else:
        df["user_agent_reuse_count"] = np.nan

    keys = ["family", "dataset", "src_ip", "dst_ip", "dst_port", "proto"]
    profile = (
        df.groupby(keys, dropna=False)
        .agg(
            http_requests_window=("ts", "count"),
            http_method=("method", mode_or_unknown) if "method" in df.columns else ("ts", lambda _s: "unknown"),
            http_status_code_median=("status_code", "median"),
            http_request_body_len_mean=("request_body_len", "mean"),
            http_response_body_len_mean=("response_body_len", "mean"),
            http_uri_length_mean=("uri_length", "mean"),
            http_uri_entropy_mean=("uri_entropy", "mean"),
            http_host_length_mean=("host_length", "mean"),
            http_user_agent_length_mean=("user_agent_length", "mean"),
            http_user_agent_reuse_mean=("user_agent_reuse_count", "mean"),
            http_get_ratio=("is_get", "mean"),
            http_post_ratio=("is_post", "mean"),
        )
        .reset_index()
    )
    total_body = profile["http_request_body_len_mean"].fillna(0) + profile["http_response_body_len_mean"].fillna(0)
    profile["http_request_response_byte_ratio"] = safe_div(
        profile["http_request_body_len_mean"].fillna(0) + 1,
        profile["http_response_body_len_mean"].fillna(0) + 1,
    )
    profile["http_request_body_fraction"] = safe_div(profile["http_request_body_len_mean"], total_body)
    return profile


http_features = build_http_channel_features(http)
print(f"Built HTTP channel features: {http_features.shape}", flush=True)
http_features.head()



Loading 43 http.log files
  loaded 1/43 http.log files


  loaded 10/43 http.log files
  loaded 20/43 http.log files
  loaded 30/43 http.log files
  loaded 40/43 http.log files
  loaded 43/43 http.log files
Loaded http rows: 148,203
Built HTTP channel features: (7722, 20)


,family,dataset,src_ip,dst_ip,dst_port,proto,http_requests_window,http_method,http_status_code_median,http_request_body_len_mean,http_response_body_len_mean,http_uri_length_mean,http_uri_entropy_mean,http_host_length_mean,http_user_agent_length_mean,http_user_agent_reuse_mean,http_get_ratio,http_post_ratio,http_request_response_byte_ratio,http_request_body_fraction
0,benign,benign_2013-12-17_capture1,10.0.0.46,173.194.112.5,80,tcp,4,GET,200.0,0.0,35.000000,538.75,5.285415,24.0,102.0,12.0,1.0,0.0,0.027778,0.000000
1,benign,benign_2013-12-17_capture1,10.0.0.46,178.255.83.1,80,tcp,2,POST,200.0,115.5,471.500000,1.0,0.000000,17.0,102.0,12.0,0.0,1.0,0.246561,0.196763
2,benign,benign_2013-12-17_capture1,10.0.0.46,199.7.54.72,80,tcp,3,POST,200.0,115.0,1925.666667,1.0,0.000000,26.0,102.0,12.0,0.0,1.0,0.060208,0.056354
3,benign,benign_2013-12-17_capture1,10.0.0.46,199.7.57.72,80,tcp,1,POST,200.0,115.0,1987.000000,1.0,0.000000,26.0,102.0,12.0,0.0,1.0,0.058350,0.054710
4,benign,benign_2013-12-17_capture1,10.0.0.46,205.201.132.35,80,tcp,1,GET,302.0,0.0,0.000000,67.0,4.358333,33.0,102.0,12.0,1.0,0.0,1.000000,NaN


## TLS and X.509 features



In [31]:
ssl_frames = []
for tls_log_type in ["ssl", "tls"]:
    frame = load_log_type(tls_log_type, inventory)
    if not frame.empty:
        ssl_frames.append(frame)
ssl = pd.concat(ssl_frames, ignore_index=True) if ssl_frames else pd.DataFrame()
x509 = load_log_type("x509", inventory)
print(f"Loaded ssl/tls rows: {len(ssl):,}; x509 rows: {len(x509):,}", flush=True)


def build_tls_channel_features(ssl_df: pd.DataFrame) -> pd.DataFrame:
    if ssl_df.empty:
        return pd.DataFrame(columns=["family", "dataset", "src_ip", "dst_ip", "dst_port", "proto"])
    df = ssl_df.rename(
        columns={
            "ts": "ts",
            "id.orig_h": "src_ip",
            "id.resp_h": "dst_ip",
            "id.resp_p": "dst_port",
        }
    ).copy()
    df["proto"] = "tcp"
    df = to_numeric(df, ["ts", "dst_port"])

    server_col = "server_name" if "server_name" in df.columns else None
    if server_col:
        df["tls_sni_present"] = df[server_col].notna().astype(float)
        df["tls_sni_length"] = df[server_col].astype("string").str.len()
        df["tls_sni_labels"] = df[server_col].map(label_count)
        df["tls_sni_entropy"] = df[server_col].map(entropy_text)
        df["tls_sni_digit_ratio"] = df[server_col].map(digit_ratio)
        df["tls_sni_reuse_count"] = df.groupby(["dataset", server_col], dropna=False)[server_col].transform("count")
    else:
        df["tls_sni_present"] = np.nan
        df["tls_sni_length"] = np.nan
        df["tls_sni_labels"] = np.nan
        df["tls_sni_entropy"] = np.nan
        df["tls_sni_digit_ratio"] = np.nan
        df["tls_sni_reuse_count"] = np.nan

    for col, out_col in [("resumed", "tls_resumed"), ("established", "tls_established")]:
        if col in df.columns:
            df[out_col] = df[col].astype("string").str.lower().isin(["t", "true"]).astype(float)
        else:
            df[out_col] = np.nan
    df["tls_cert_chain_length"] = df["cert_chain_fuids"].map(list_count) if "cert_chain_fuids" in df.columns else np.nan

    keys = ["family", "dataset", "src_ip", "dst_ip", "dst_port", "proto"]
    profile = (
        df.groupby(keys, dropna=False)
        .agg(
            tls_sessions_window=("ts", "count"),
            tls_version=("version", mode_or_unknown) if "version" in df.columns else ("ts", lambda _s: "unknown"),
            tls_cipher=("cipher", mode_or_unknown) if "cipher" in df.columns else ("ts", lambda _s: "unknown"),
            tls_curve=("curve", mode_or_unknown) if "curve" in df.columns else ("ts", lambda _s: "unknown"),
            tls_next_protocol=("next_protocol", mode_or_unknown) if "next_protocol" in df.columns else ("ts", lambda _s: "unknown"),
            tls_resumed_ratio=("tls_resumed", "mean"),
            tls_established_ratio=("tls_established", "mean"),
            tls_sni_present_ratio=("tls_sni_present", "mean"),
            tls_sni_length_mean=("tls_sni_length", "mean"),
            tls_sni_label_count_mean=("tls_sni_labels", "mean"),
            tls_sni_entropy_mean=("tls_sni_entropy", "mean"),
            tls_sni_digit_ratio_mean=("tls_sni_digit_ratio", "mean"),
            tls_sni_reuse_count_mean=("tls_sni_reuse_count", "mean"),
            tls_cert_chain_length_mean=("tls_cert_chain_length", "mean"),
        )
        .reset_index()
    )
    return profile


def build_x509_channel_features(ssl_df: pd.DataFrame, x509_df: pd.DataFrame) -> pd.DataFrame:
    if ssl_df.empty or x509_df.empty or "cert_chain_fuids" not in ssl_df.columns:
        return pd.DataFrame(columns=["family", "dataset", "src_ip", "dst_ip", "dst_port", "proto"])

    ssl_base = ssl_df.rename(
        columns={
            "ts": "ts",
            "id.orig_h": "src_ip",
            "id.resp_h": "dst_ip",
            "id.resp_p": "dst_port",
        }
    ).copy()
    ssl_base["proto"] = "tcp"
    ssl_base = to_numeric(ssl_base, ["ts", "dst_port"])
    ssl_base["cert_id"] = ssl_base["cert_chain_fuids"].astype("string").str.split(",")
    ssl_base = ssl_base.explode("cert_id")
    ssl_base["cert_id"] = ssl_base["cert_id"].replace({"<NA>": np.nan})
    ssl_base = ssl_base.dropna(subset=["cert_id"])

    cert = x509_df.copy()
    if "id" not in cert.columns:
        return pd.DataFrame(columns=["family", "dataset", "src_ip", "dst_ip", "dst_port", "proto"])
    cert = cert.rename(columns={"id": "cert_id"})
    cert = to_numeric(
        cert,
        [
            "certificate.not_valid_before",
            "certificate.not_valid_after",
            "certificate.key_length",
        ],
    )
    cert["cert_lifetime_days"] = (
        cert["certificate.not_valid_after"] - cert["certificate.not_valid_before"]
    ) / 86400
    cert["cert_subject_length"] = cert["certificate.subject"].astype("string").str.len() if "certificate.subject" in cert.columns else np.nan
    cert["cert_issuer_length"] = cert["certificate.issuer"].astype("string").str.len() if "certificate.issuer" in cert.columns else np.nan
    if {"certificate.subject", "certificate.issuer"}.issubset(cert.columns):
        cert["cert_self_signed"] = cert["certificate.subject"].eq(cert["certificate.issuer"]).astype(float)
    else:
        cert["cert_self_signed"] = np.nan
    san_cols = [col for col in cert.columns if "san" in col.lower()]
    cert["cert_san_count"] = cert[san_cols[0]].map(list_count) if san_cols else np.nan

    merged = ssl_base.merge(
        cert,
        on=["dataset", "cert_id"],
        how="left",
        suffixes=("", "_cert"),
    )
    merged["cert_age_days"] = (merged["ts"] - merged["certificate.not_valid_before"]) / 86400
    keys = ["family", "dataset", "src_ip", "dst_ip", "dst_port", "proto"]
    profile = (
        merged.groupby(keys, dropna=False)
        .agg(
            cert_lifetime_days_mean=("cert_lifetime_days", "mean"),
            cert_age_days_mean=("cert_age_days", "mean"),
            cert_key_length_mean=("certificate.key_length", "mean"),
            cert_subject_length_mean=("cert_subject_length", "mean"),
            cert_issuer_length_mean=("cert_issuer_length", "mean"),
            cert_self_signed_ratio=("cert_self_signed", "mean"),
            cert_san_count_mean=("cert_san_count", "mean"),
            cert_key_alg=("certificate.key_alg", mode_or_unknown) if "certificate.key_alg" in merged.columns else ("cert_id", lambda _s: "unknown"),
            cert_sig_alg=("certificate.sig_alg", mode_or_unknown) if "certificate.sig_alg" in merged.columns else ("cert_id", lambda _s: "unknown"),
            cert_key_type=("certificate.key_type", mode_or_unknown) if "certificate.key_type" in merged.columns else ("cert_id", lambda _s: "unknown"),
        )
        .reset_index()
    )
    return profile


tls_features = build_tls_channel_features(ssl)
x509_features = build_x509_channel_features(ssl, x509)
print(f"Built TLS channel features: {tls_features.shape}; X509 channel features: {x509_features.shape}", flush=True)
tls_features.head()



Loading 51 ssl.log files


  loaded 1/51 ssl.log files
  loaded 10/51 ssl.log files
  loaded 20/51 ssl.log files
  loaded 30/51 ssl.log files
  loaded 40/51 ssl.log files
  loaded 50/51 ssl.log files
  loaded 51/51 ssl.log files
Loading 0 tls.log files
Loading 21 x509.log files
  loaded 1/21 x509.log files
  loaded 10/21 x509.log files
  loaded 20/21 x509.log files
  loaded 21/21 x509.log files
Loaded ssl/tls rows: 359,984; x509 rows: 4,058
Built TLS channel features: (39289, 20); X509 channel features: (0, 6)


,family,dataset,src_ip,dst_ip,dst_port,proto,tls_sessions_window,tls_version,tls_cipher,tls_curve,tls_next_protocol,tls_resumed_ratio,tls_established_ratio,tls_sni_present_ratio,tls_sni_length_mean,tls_sni_label_count_mean,tls_sni_entropy_mean,tls_sni_digit_ratio_mean,tls_sni_reuse_count_mean,tls_cert_chain_length_mean
0,benign,benign1_smashburger,192.168.46.84,13.225.239.11,443,tcp,2,TLSv13,TLS_AES_128_GCM_SHA256,x25519,unknown,0.0,1.0,1.0,15.0,2.0,3.506891,0.0,11.0,NaN
1,benign,benign1_smashburger,192.168.46.84,13.225.239.120,443,tcp,2,TLSv13,TLS_AES_128_GCM_SHA256,x25519,unknown,0.0,1.0,1.0,15.0,2.0,3.506891,0.0,11.0,NaN
2,benign,benign1_smashburger,192.168.46.84,13.225.239.53,443,tcp,3,TLSv13,TLS_AES_128_GCM_SHA256,x25519,unknown,0.0,1.0,1.0,15.0,2.0,3.506891,0.0,11.0,NaN
3,benign,benign1_smashburger,192.168.46.84,13.225.239.61,443,tcp,4,TLSv13,TLS_AES_128_GCM_SHA256,x25519,unknown,0.0,1.0,1.0,15.0,2.0,3.506891,0.0,11.0,NaN
4,benign,benign2_smashburger,192.168.46.84,13.225.239.11,443,tcp,5,TLSv13,TLS_AES_128_GCM_SHA256,x25519,unknown,0.0,1.0,1.0,15.0,2.0,3.506891,0.0,11.0,NaN


## Merge model-ready channel dataset



In [32]:
channel_keys = ["family", "dataset", "src_ip", "dst_ip", "dst_port", "proto"]
model_df = channel_features.copy()
model_df = model_df.merge(dns_features, on=["family", "dataset", "src_ip"], how="left")
model_df = model_df.merge(http_features, on=channel_keys, how="left")
model_df = model_df.merge(tls_features, on=channel_keys, how="left")
model_df = model_df.merge(x509_features, on=channel_keys, how="left")

model_df = model_df[model_df["family"].isin(ALL_FAMILIES)].copy()
model_df["binary_label"] = model_df["family"].ne("benign").astype(int)
model_df["multiclass_label"] = model_df["family"]

model_df.to_csv(OUTPUT_DIR / "model_ready_channel_features.csv", index=False)
print(f"Model-ready channel dataset: {model_df.shape}", flush=True)
model_df[["family", "dataset", "src_ip", "dst_ip", "dst_port", "proto", "connections_window", "binary_label"]].head()



Model-ready channel dataset: (54200, 92)


,family,dataset,src_ip,dst_ip,dst_port,proto,connections_window,binary_label
0,benign,benign1_smashburger,192.168.46.84,13.225.239.11,443,tcp,2,0
1,benign,benign1_smashburger,192.168.46.84,13.225.239.120,443,tcp,2,0
2,benign,benign1_smashburger,192.168.46.84,13.225.239.53,443,tcp,3,0
3,benign,benign1_smashburger,192.168.46.84,13.225.239.61,443,tcp,4,0
4,benign,benign2_smashburger,192.168.46.84,13.225.239.11,443,tcp,5,0


In [33]:
dataset_summary = (
    model_df.groupby(["family", "dataset"], as_index=False)
    .agg(
        channels=("binary_label", "size"),
        unique_src=("src_ip", "nunique"),
        unique_dst=("dst_ip", "nunique"),
        median_connections_per_channel=("connections_window", "median"),
    )
    .sort_values(["family", "dataset"])
)
dataset_summary.to_csv(OUTPUT_DIR / "dataset_summary.csv", index=False)
dataset_summary



,family,dataset,channels,unique_src,unique_dst,median_connections_per_channel
0,benign,benign1_smashburger,4,1,4,2.5
1,benign,benign2_smashburger,3,1,3,5.0
2,benign,benign3_smashburger,1,1,1,1.0
3,benign,benign4_smashburger,4,1,4,1.5
4,benign,benign5_smashburger,3,1,3,1.0
...,...,...,...,...,...,...
70,sliver,sliver_A_1148,1441,47,65,2.0
71,sliver,sliver_A_1358,1483,47,61,3.0
72,sliver,sliver_A_1607,1495,47,64,2.0
73,sliver,sliver_A_1815,2001,47,82,5.0


## Feature columns



In [34]:
leakage_columns = {
    "src_ip",
    "dst_ip",
    "start_ts",
    "end_ts",
    "source_file",
    "binary_label",
    "multiclass_label",
    "is_c2",
}
metadata_columns = {"family", "dataset"}

candidate_cols = [
    col
    for col in model_df.columns
    if col not in leakage_columns and col not in metadata_columns
    and not model_df[col].isna().all()
]

numeric_features = [col for col in candidate_cols if is_numeric_dtype(model_df[col])]
categorical_features = [col for col in candidate_cols if col not in numeric_features]

feature_register = pd.DataFrame(
    [{"feature": col, "type": "categorical" if col in categorical_features else "numeric"} for col in candidate_cols]
)
feature_register.to_csv(OUTPUT_DIR / "candidate_model_features.csv", index=False)
print(f"Numeric features: {len(numeric_features)}; categorical features: {len(categorical_features)}", flush=True)
feature_register.head(80)



Numeric features: 72; categorical features: 10


,feature,type
0,dst_port,numeric
1,proto,categorical
2,connections_window,numeric
3,service,categorical
4,conn_state,categorical
...,...,...
75,tls_established_ratio,numeric
76,tls_sni_present_ratio,numeric
77,tls_sni_length_mean,numeric
78,tls_sni_label_count_mean,numeric


## Train/test split helpers



In [35]:
def split_datasets_by_family(
    df: pd.DataFrame,
    test_fraction: float = 0.35,
) -> tuple[set[str], set[str]]:
    train_datasets = set()
    test_datasets = set()

    min_test_rows = {"benign": 1_000}
    min_train_rows = {"benign": 1_000}
    max_test_fraction = {"benign": 0.50}

    for family, group in df.groupby("family"):
        sizes = (
            group.groupby("dataset")
            .size()
            .rename("rows")
            .reset_index()
            .sort_values("dataset")
        )

        datasets = set(sizes["dataset"].astype(str))

        if len(sizes) == 1:
            train_datasets.update(datasets)
            continue

        total_rows = int(sizes["rows"].sum())
        target_rows = int(math.ceil(total_rows * test_fraction))
        target_rows = max(target_rows, min_test_rows.get(family, 1))

        min_train = min_train_rows.get(family, 1)
        max_test_rows = int(total_rows * max_test_fraction.get(family, 0.95))

        selected = []
        selected_rows = 0

        shuffled = sizes.sample(frac=1.0, random_state=RANDOM_STATE)

        for _, row in shuffled.iterrows():
            dataset = str(row["dataset"])
            rows = int(row["rows"])

            if selected_rows + rows > max_test_rows:
                continue

            if total_rows - (selected_rows + rows) < min_train:
                continue

            selected.append(dataset)
            selected_rows += rows

            if selected_rows >= target_rows:
                break

        test = set(selected)
        train = datasets - test

        if not test:
            test.add(str(sizes.sample(1, random_state=RANDOM_STATE).iloc[0]["dataset"]))
            train = datasets - test

        if not train:
            largest_test_dataset = (
                sizes[sizes["dataset"].astype(str).isin(test)]
                .sort_values("rows", ascending=False)
                .iloc[0]["dataset"]
            )
            test.remove(str(largest_test_dataset))
            train.add(str(largest_test_dataset))

        train_datasets.update(train)
        test_datasets.update(test)

    return train_datasets, test_datasets


TRAIN_DATASETS, TEST_DATASETS = split_datasets_by_family(model_df)

present_datasets = set(model_df["dataset"].unique())

forced_train = FORCE_TRAIN_DATASETS & present_datasets
forced_test = FORCE_TEST_DATASETS & present_datasets

TRAIN_DATASETS = (TRAIN_DATASETS - forced_test) | forced_train
TEST_DATASETS = (TEST_DATASETS - forced_train) | forced_test

print(f"Train datasets: {len(TRAIN_DATASETS)}; test datasets: {len(TEST_DATASETS)}", flush=True)

split_summary = (
    model_df.assign(split=np.where(model_df["dataset"].isin(TEST_DATASETS), "test", "train"))
    .groupby(["family", "split"], as_index=False)
    .agg(channels=("binary_label", "size"), captures=("dataset", "nunique"))
    .sort_values(["family", "split"])
)

split_summary.to_csv(OUTPUT_DIR / "train_test_split_summary.csv", index=False)
split_summary

def make_preprocessor() -> Pipeline:
    numeric_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=3)),
        ]
    )

    preprocessor = ColumnTransformer(
        [
            ("num", numeric_pipe, numeric_features),
            ("cat", categorical_pipe, categorical_features),
        ]
    )

    return Pipeline(
        [
            ("preprocess", preprocessor),
            ("select", SelectKBest(score_func=f_classif, k=min(TOP_K_FEATURES, len(candidate_cols)))),
        ]
    )


MODELS = {
    "logistic_regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "decision_tree": DecisionTreeClassifier(
        max_depth=7,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=120,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

MODEL_NAMES = list(MODELS)


def build_model(model_name: str) -> Pipeline:
    return Pipeline(
        [
            *make_preprocessor().steps,
            ("model", MODELS[model_name]),
        ]
    )


def model_feature_names(pipe: Pipeline) -> np.ndarray:
    names = pipe.named_steps["preprocess"].get_feature_names_out()
    mask = pipe.named_steps["select"].get_support()

    return names[mask]


def selected_feature_table(pipe: Pipeline, experiment: str, model_name: str) -> pd.DataFrame:
    selector = pipe.named_steps["select"]
    names = model_feature_names(pipe)
    scores = selector.scores_[selector.get_support()]

    return (
        pd.DataFrame(
            {
                "experiment": experiment,
                "model": model_name,
                "feature": names,
                "selection_score": scores,
            }
        )
        .sort_values("selection_score", ascending=False)
        .reset_index(drop=True)
    )

Train datasets: 36; test datasets: 39


## Experiment 1: Binary detection



In [ ]:
binary_train = model_df[model_df["dataset"].isin(TRAIN_DATASETS)].copy()
binary_test = model_df[model_df["dataset"].isin(TEST_DATASETS)].copy()

binary_results = []
selected_features = []

for model_name in MODEL_NAMES:
    print(f"Binary detection: training {model_name}", flush=True)

    model = build_model(model_name)
    model.fit(binary_train[candidate_cols], binary_train["binary_label"])

    y_true = binary_test["binary_label"]
    y_pred = model.predict(binary_test[candidate_cols])
    y_score = model.predict_proba(binary_test[candidate_cols])[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    binary_results.append(
        {
            "model": model_name,
            "train_rows": len(binary_train),
            "test_rows": len(binary_test),
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "fpr": fp / (fp + tn) if fp + tn else np.nan,
            "roc_auc": roc_auc_score(y_true, y_score),
            "pr_auc": average_precision_score(y_true, y_score),
        }
    )

    selected_features.append(selected_feature_table(model, "binary_detection", model_name))

binary_results_df = pd.DataFrame(binary_results).sort_values("f1", ascending=False)
binary_results_df.to_csv(OUTPUT_DIR / "binary_detection_results.csv", index=False)

binary_results_df

Binary detection: training logistic_regression


## Experiment 2: Multiclass attribution



In [ ]:
multiclass_results = []
multiclass_predictions = []

for model_name in MODEL_NAMES:
    print(f"Multiclass attribution: training {model_name}", flush=True)
    model = build_model(model_name)
    model.fit(binary_train[candidate_cols], binary_train["multiclass_label"])
    pred = model.predict(binary_test[candidate_cols])
    report = classification_report(binary_test["multiclass_label"], pred, output_dict=True, zero_division=0)
    row = {
        "experiment": "multiclass_attribution",
        "model": model_name,
        "train_rows": len(binary_train),
        "test_rows": len(binary_test),
        "accuracy": accuracy_score(binary_test["multiclass_label"], pred),
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
    }
    for family in ALL_FAMILIES:
        if family in report:
            row[f"{family}_precision"] = report[family]["precision"]
            row[f"{family}_recall"] = report[family]["recall"]
            row[f"{family}_f1"] = report[family]["f1-score"]
    multiclass_results.append(row)
    selected_features.append(selected_feature_table(model, "multiclass_attribution", model_name))
    preds = binary_test[["family", "dataset", "src_ip", "dst_ip", "dst_port", "proto", "multiclass_label"]].copy()
    preds["model"] = model_name
    preds["prediction"] = pred
    multiclass_predictions.append(preds)

multiclass_results_df = pd.DataFrame(multiclass_results).sort_values("macro_f1", ascending=False)
multiclass_results_df.to_csv(OUTPUT_DIR / "multiclass_attribution_results.csv", index=False)
pd.concat(multiclass_predictions, ignore_index=True).to_csv(OUTPUT_DIR / "multiclass_attribution_predictions.csv", index=False)
multiclass_results_df



Multiclass attribution: training logistic_regression
Multiclass attribution: training decision_tree
Multiclass attribution: training random_forest


,experiment,model,train_rows,test_rows,accuracy,macro_f1,weighted_f1,benign_precision,benign_recall,benign_f1,cobalt_strike_precision,cobalt_strike_recall,cobalt_strike_f1,havoc_precision,havoc_recall,havoc_f1,mythic_precision,mythic_recall,mythic_f1,sliver_precision,sliver_recall,sliver_f1
2,multiclass_attribution,random_forest,43600,10600,0.979245,0.887115,0.977326,0.989711,0.998024,0.993850,0.855368,0.969819,0.909005,0.816176,0.431907,0.564885,0.976645,0.962343,0.969442,1.000000,0.996787,0.998391
0,multiclass_attribution,logistic_regression,43600,10600,0.968679,0.858671,0.967037,0.991580,0.989130,0.990354,0.851175,0.983903,0.912739,0.622642,0.385214,0.475962,0.876404,0.979079,0.924901,0.997329,0.981598,0.989401
1,multiclass_attribution,decision_tree,43600,10600,0.951604,0.814855,0.955311,1.000000,0.988142,0.994036,0.858527,0.891348,0.874630,0.309192,0.431907,0.360390,0.795737,0.937238,0.860711,0.999398,0.970060,0.984510


In [ ]:
best_multiclass_model_name = multiclass_results_df.iloc[0]["model"]
best_multiclass = build_model(best_multiclass_model_name)
best_multiclass.fit(binary_train[candidate_cols], binary_train["multiclass_label"])
best_multi_pred = best_multiclass.predict(binary_test[candidate_cols])
cm = confusion_matrix(binary_test["multiclass_label"], best_multi_pred, labels=ALL_FAMILIES)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=ALL_FAMILIES, yticklabels=ALL_FAMILIES, cmap="Blues")
plt.title(f"Multiclass confusion matrix: {best_multiclass_model_name}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "multiclass_confusion_matrix.png", dpi=160)
plt.close()



## Experiment 3: Leave-one-C2-family-out generalisation



In [ ]:
def make_unseen_split(df: pd.DataFrame, unknown_family: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    benign_train = df["family"].eq("benign") & df["dataset"].isin(TRAIN_DATASETS)
    known_c2_train = df["family"].isin([fam for fam in C2_FAMILIES if fam != unknown_family])
    unknown_test = df["family"].eq(unknown_family)
    benign_test = df["family"].eq("benign") & df["dataset"].isin(TEST_DATASETS)
    train = df[benign_train | known_c2_train].copy()
    test = df[unknown_test | benign_test].copy()
    return train, test


unseen_results = []
unseen_predictions = []

for unknown_family in C2_FAMILIES:
    train_df, test_df = make_unseen_split(model_df, unknown_family)
    if train_df["binary_label"].nunique() < 2 or test_df["binary_label"].nunique() < 2:
        print(f"Skipping {unknown_family}: insufficient class coverage")
        continue
    for model_name in MODEL_NAMES:
        print(f"Unseen C2 {unknown_family}: training {model_name}", flush=True)
        model = build_model(model_name)
        model.fit(train_df[candidate_cols], train_df["binary_label"])
        pred = model.predict(test_df[candidate_cols])
        proba = model.predict_proba(test_df[candidate_cols])[:, 1]
        tn, fp, fn, tp = confusion_matrix(test_df["binary_label"], pred, labels=[0, 1]).ravel()
        unseen_results.append(
            {
                "experiment": "leave_one_c2_family_out",
                "unknown_family": unknown_family,
                "model": model_name,
                "known_train_families": ",".join(["benign"] + [fam for fam in C2_FAMILIES if fam != unknown_family]),
                "train_rows": len(train_df),
                "test_rows": len(test_df),
                "unknown_test_rows": int(test_df["family"].eq(unknown_family).sum()),
                "benign_test_rows": int(test_df["family"].eq("benign").sum()),
                "tn": tn,
                "fp": fp,
                "fn": fn,
                "tp": tp,
                "accuracy": accuracy_score(test_df["binary_label"], pred),
                "precision": precision_score(test_df["binary_label"], pred, zero_division=0),
                "recall": recall_score(test_df["binary_label"], pred, zero_division=0),
                "f1": f1_score(test_df["binary_label"], pred, zero_division=0),
                "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
                "roc_auc": roc_auc_score(test_df["binary_label"], proba),
                "pr_auc": average_precision_score(test_df["binary_label"], proba),
            }
        )
        selected_features.append(selected_feature_table(model, f"unseen_{unknown_family}", model_name))
        preds = test_df[["family", "dataset", "src_ip", "dst_ip", "dst_port", "proto", "binary_label"]].copy()
        preds["unknown_family"] = unknown_family
        preds["model"] = model_name
        preds["prediction"] = pred
        preds["score"] = proba
        unseen_predictions.append(preds)

unseen_results_df = pd.DataFrame(unseen_results).sort_values(["unknown_family", "f1"], ascending=[True, False])
unseen_results_df.to_csv(OUTPUT_DIR / "leave_one_c2_family_out_results.csv", index=False)
pd.concat(unseen_predictions, ignore_index=True).to_csv(OUTPUT_DIR / "leave_one_c2_family_out_predictions.csv", index=False)
unseen_results_df



Unseen C2 cobalt_strike: training logistic_regression
Unseen C2 cobalt_strike: training decision_tree
Unseen C2 cobalt_strike: training random_forest
Unseen C2 havoc: training logistic_regression
Unseen C2 havoc: training decision_tree
Unseen C2 havoc: training random_forest
Unseen C2 mythic: training logistic_regression
Unseen C2 mythic: training decision_tree
Unseen C2 mythic: training random_forest
Unseen C2 sliver: training logistic_regression
Unseen C2 sliver: training decision_tree
Unseen C2 sliver: training random_forest


,experiment,unknown_family,model,known_train_families,train_rows,test_rows,unknown_test_rows,benign_test_rows,tn,fp,fn,tp,accuracy,precision,recall,f1,fpr,roc_auc,pr_auc
2,leave_one_c2_family_out,cobalt_strike,random_forest,"benign,havoc,mythic,sliver",49900,4300,2276,2024,2008,16,59,2217,0.982558,0.992835,0.974077,0.983367,0.007905,0.998340,0.998266
0,leave_one_c2_family_out,cobalt_strike,logistic_regression,"benign,havoc,mythic,sliver",49900,4300,2276,2024,2002,22,145,2131,0.961163,0.989782,0.936292,0.962294,0.010870,0.996139,0.996590
1,leave_one_c2_family_out,cobalt_strike,decision_tree,"benign,havoc,mythic,sliver",49900,4300,2276,2024,2005,19,167,2109,0.956744,0.991071,0.926626,0.957766,0.009387,0.965369,0.964352
5,leave_one_c2_family_out,havoc,random_forest,"benign,cobalt_strike,mythic,sliver",50474,3726,1702,2024,2008,16,371,1331,0.896135,0.988122,0.782021,0.873073,0.007905,0.996319,0.994411
3,leave_one_c2_family_out,havoc,logistic_regression,"benign,cobalt_strike,mythic,sliver",50474,3726,1702,2024,2004,20,370,1332,0.895330,0.985207,0.782609,0.872299,0.009881,0.990638,0.990038
4,leave_one_c2_family_out,havoc,decision_tree,"benign,cobalt_strike,mythic,sliver",50474,3726,1702,2024,2009,15,547,1155,0.849168,0.987179,0.678613,0.804318,0.007411,0.901090,0.886412
8,leave_one_c2_family_out,mythic,random_forest,"benign,cobalt_strike,havoc,sliver",50811,3389,1365,2024,2008,16,12,1353,0.991738,0.988313,0.991209,0.989759,0.007905,0.999415,0.998263
7,leave_one_c2_family_out,mythic,decision_tree,"benign,cobalt_strike,havoc,sliver",50811,3389,1365,2024,2009,15,14,1351,0.991443,0.989019,0.989744,0.989381,0.007411,0.996218,0.988919
6,leave_one_c2_family_out,mythic,logistic_regression,"benign,cobalt_strike,havoc,sliver",50811,3389,1365,2024,2004,20,23,1342,0.987312,0.985316,0.983150,0.984232,0.009881,0.997191,0.995178
10,leave_one_c2_family_out,sliver,decision_tree,"benign,cobalt_strike,havoc,mythic",32614,21586,19562,2024,2001,23,70,19492,0.995692,0.998821,0.996422,0.997620,0.011364,0.991405,0.998527


## Feature selection summary



In [ ]:
selected_features_df = pd.concat(selected_features, ignore_index=True)
selected_features_df.to_csv(OUTPUT_DIR / "selected_features.csv", index=False)

top_selected = (
    selected_features_df.assign(base_feature=lambda d: d["feature"].str.replace(r"^(num|cat)__", "", regex=True))
    .groupby("base_feature", as_index=False)
    .agg(
        times_selected=("base_feature", "size"),
        mean_selection_score=("selection_score", "mean"),
    )
    .sort_values(["times_selected", "mean_selection_score"], ascending=False)
)
top_selected.to_csv(OUTPUT_DIR / "selected_feature_frequency.csv", index=False)
top_selected.head(40)



,base_feature,times_selected,mean_selection_score
75,unique_destinations_window,18,139663.559966
47,source_connections_window,18,70932.270740
74,tls_version_TLSv13,18,61960.428338
50,tls_cipher_TLS_AES_256_GCM_SHA384,18,17363.252737
71,tls_sni_reuse_count_mean,18,15570.587556
3,conn_state_REJ,18,9712.627895
58,tls_curve_secp256r1,18,8607.448409
22,dns_ttl_std,18,8151.464872
0,active_span_seconds,18,7994.584552
2,bytes_per_packet,18,7277.350290
